**Sample ID**: 284_base_US_ToolShift

**Query**:

Can you help me with resolving some of the assigned issues of Musa.

**DB Type**: Base Case

**Case Description**:

The Jira workspace contains three projects: "WEBAPP", "MOBILE", and "BACKEND", with available issue statuses including "In Progress", "Open", and "Resolved". User "musa" has five issues assigned across these projects - three issues with priority "High" currently having status "In Progress", and two issues with priority "Medium" having status "Open". All high priority issues are of type "Bug". The Slack workspace contains a channel named "dev-team" for development notifications with no notifications about resolved issues.

```
<multiturn info>
[turn 1]: Clarification on the issues: "High" Priority (Information Gathering)
[turn 2]: Query 2: Now, create a summary report in Confluence (Follow Up Request)
[turn 3]: Instead, notify the dev team on Slack about the resolved issues with their ids along with the total count of resolved issues(Goal Shift)
</multiturn info>
```

```
<tools>
[turn 0]: jira
[turn 2]: confluence
[turn 3]: slack
</tools>
```

**Global/Context Variables:**


**APIs:**

- jira
- slack
- confluence


# Set Up

## Download relevant files

In [ ]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # This will be replaced dynamically

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")

# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
                if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")

# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.6 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.6.zip (ID: 1XQql-2xKE-aw6rvzsv177b1Co-E4D2u8)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.6.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.


## Install Dependencies and Clone Repositories

In [ ]:
!pip install -r /content/APIs/requirements.txt

## Import APIs and initiate DBs

In [ ]:
# proto_ignore
import random
import sys
import uuid
import secrets

# Import libraries to ensure all initializations by the python libraries are complete
import jira
import slack
import confluence

def patch_randomness(seed=42):
    rng = random.Random(seed)
    random.seed(seed)

    # Patch uuid.uuid4
    def deterministic_uuid4():
        return uuid.UUID(int=rng.getrandbits(128))
    sys.modules['uuid'].uuid4 = deterministic_uuid4

    # Patch secrets to use the same deterministic random generator
    class DeterministicRandom:
        def randbelow(self, n):
            return rng.randrange(n)

        def choice(self, seq):
            return rng.choice(seq)

        def randbits(self, k):
            return rng.getrandbits(k)

        def randint(self, a, b):
            return rng.randint(a, b)
    sys.modules['secrets'] = DeterministicRandom()

patch_randomness()

In [ ]:
import jira
import slack
import confluence

# Load default databases
jira.SimulationEngine.db.load_state("/content/DBs/JiraDefaultDB.json")
slack.SimulationEngine.db.load_state("/content/DBs/SlackDefaultDB.json")
confluence.SimulationEngine.db.load_state("/content/DBs/ConfluenceDefaultDB.json")

print("--- Jira and Slack Initial State Setup ---")

# --- Jira Setup ---

# 1. Create Users
print("Creating Jira users...")
user_musa_payload = {
    "name": "musa",
    "emailAddress": "musa.k@appgenix.com",
    "displayName": "Musa Khan"
}
user_liam_payload = {
    "name": "liam",
    "emailAddress": "liam.n@appgenix.com",
    "displayName": "Liam Neeson"
}
user_musa = jira.create_user(user_musa_payload)
print(f"Created Jira user: {user_musa.get('user', {}).get('name', '')}")
user_liam = jira.create_user(user_liam_payload)
print(f"Created Jira user: {user_liam.get('user', {}).get('name', '')}")


# 2. Create Projects
print("\nCreating Jira projects...")
projects_to_create = [
    {"key": "WEBAPP", "name": "Web Application"},
    {"key": "MOBILE", "name": "Mobile App"},
    {"key": "BACKEND", "name": "Backend Services"}
]
for project in projects_to_create:
    created_project = jira.create_project(proj_key=project["key"], proj_name=project["name"])
    if created_project.get('created'):
        print(f"Created project: {created_project.get('project', {}).get('key', '')}")

# 3. Create Issues
print("\nCreating Jira issues...")

# Issues for Musa that match the criteria
issues_for_musa = [
    # 3 High priority, In Progress, Bug issues
    {
        "project": "WEBAPP", "summary": "UI glitch on login screen", "description": "The login button is misaligned on Firefox.",
        "issuetype": "Bug", "priority": "High", "status": "In Progress", "assignee": {"name": "musa"}
    },
    {
        "project": "MOBILE", "summary": "App freezes on loading user profile", "description": "The application becomes unresponsive when a user profile is loaded.",
        "issuetype": "Bug", "priority": "High", "status": "In Progress", "assignee": {"name": "musa"}
    },
    {
        "project": "BACKEND", "summary": "Authentication service timeout", "description": "The authentication service times out under heavy load.",
        "issuetype": "Bug", "priority": "High", "status": "In Progress", "assignee": {"name": "musa"}
    },
    # 2 Medium priority, Open issues
    {
        "project": "WEBAPP", "summary": "Implement password reset feature", "description": "Users need a way to reset their passwords.",
        "issuetype": "Story", "priority": "Medium", "status": "Open", "assignee": {"name": "musa"}
    },
    {
        "project": "MOBILE", "summary": "Add push notification support", "description": "Enable push notifications for new messages.",
        "issuetype": "Task", "priority": "Medium", "status": "Open", "assignee": {"name": "musa"}
    }
]

# Issues that do not match the criteria
other_issues = [
    {
        "project": "BACKEND", "summary": "Optimize database query performance", "description": "Certain queries are running slower than expected.",
        "issuetype": "Task", "priority": "Low", "status": "To Do", "assignee": {"name": "liam"}
    },
    {
        "project": "WEBAPP", "summary": "Update terms of service page", "description": "The terms of service page needs to be updated with the latest legal text.",
        "issuetype": "Task", "priority": "Lowest", "status": "Open", "assignee": {"name": "liam"}
    },
    {
        "project": "MOBILE", "summary": "Investigate battery drain issue", "description": "Users are reporting excessive battery drain.",
        "issuetype": "Bug", "priority": "Medium", "status": "In Progress", "assignee": {"name": "liam"}
    }
]

all_issues_to_create = issues_for_musa + other_issues

for issue_fields in all_issues_to_create:
    created_issue = jira.create_issue(fields=issue_fields)
    if created_issue.get('id'):
        print(f"Created issue '{created_issue.get('fields', {}).get('summary', '')}' with ID: {created_issue.get('id', '')}")

# --- Slack Setup ---

# 1. Create User
print("\nCreating Slack user...")
# Invite the user first, as direct creation is not supported.
invited_user = slack.invite_admin_user(email="musa.k@appgenix.com", real_name="Musa Khan")
if invited_user.get('ok'):
    musa_slack_user_id = invited_user.get('user', {}).get('id')
    print(f"Invited and created Slack user 'Musa Khan' with ID: {musa_slack_user_id}")
else:
    print("Failed to create Slack user.")


# 2. Create Channel
print("\nCreating Slack channel...")
dev_team_channel = slack.create_channel(name="dev-team")
if dev_team_channel.get('ok'):
    channel_id = dev_team_channel.get('channel', {}).get('id')
    print(f"Created Slack channel 'dev-team' with ID: {channel_id}")
else:
    print("Failed to create Slack channel 'dev-team'.")


print("\n--- Initial State Setup Complete ---")

--- Jira and Slack Initial State Setup ---
Creating Jira users...
Created Jira user: musa
Created Jira user: liam

Creating Jira projects...
Created project: WEBAPP
Created project: MOBILE
Created project: BACKEND

Creating Jira issues...
Created issue 'UI glitch on login screen' with ID: ISSUE-4
Created issue 'App freezes on loading user profile' with ID: ISSUE-5
Created issue 'Authentication service timeout' with ID: ISSUE-6
Created issue 'Implement password reset feature' with ID: ISSUE-7
Created issue 'Add push notification support' with ID: ISSUE-8
Created issue 'Optimize database query performance' with ID: ISSUE-9
Created issue 'Update terms of service page' with ID: ISSUE-10
Created issue 'Investigate battery drain issue' with ID: ISSUE-11

Creating Slack user...
Invited and created Slack user 'Musa Khan' with ID: U0B092E45

Creating Slack channel...
Created Slack channel 'dev-team' with ID: C356E134F

--- Initial State Setup Complete ---


# Initial Assertion

1. Assert that user "musa" has exactly three issues with priority "High" and status not "Resolved" across all projects.
2. Assert that the Slack channel "dev-team" exists containing no message with issue ids of high priority unresolved issues.

In [ ]:
import jira
import slack

# 1. Assert that user "musa" has exactly three issues with priority "High" and status not "Resolved".
jql_query = "assignee = 'musa' AND priority = 'High' AND status != 'Resolved'"
search_results = jira.search_issues_jql(jql=jql_query)
unresolved_issues = search_results.get('issues', [])
assert search_results.get('total', 0) == 3, "Assertion Failed: User 'musa' should have exactly 3 high priority, unresolved issues."

# 2. Assert that the Slack channel "dev-team" exists and contains no messages with the issue IDs.
target_channel_name = "dev-team"
message_with_issue_id_found = False
dev_team_channel_id = None

all_channels = slack.list_channels(types="public_channel").get('channels', [])
for channel in all_channels:
    if channel.get('name') == target_channel_name:
        dev_team_channel_id = channel.get('id')
        break

if dev_team_channel_id:
    issue_ids = [issue.get('id') for issue in unresolved_issues]
    if issue_ids:
        history_response = slack.get_conversation_history(channel=dev_team_channel_id, limit=20)
        messages = history_response.get('messages', [])
        for msg in messages:
            message_text = msg.get('text', '')
            if any(issue_id in message_text for issue_id in issue_ids):
                message_with_issue_id_found = True
                break

assert dev_team_channel_id is not None and not message_with_issue_id_found, "Assertion Failed: Slack channel 'dev-team' should exist and contain no messages about high priority unresolved issues."

# Action

**Simulated User**: Can you help me with resolving some of the assigned issues of Musa.

In [ ]:
# proto_ignore
import jira

In [ ]:
jira.search_issues_jql(jql="assignee = 'musa'")

{'issues': [{'id': 'ISSUE-4',
   'fields': {'project': 'WEBAPP',
    'summary': 'UI glitch on login screen',
    'issuetype': 'Bug',
    'description': 'The login button is misaligned on Firefox.',
    'priority': 'High',
    'assignee': {'name': 'musa'},
    'status': 'In Progress',
    'created': '2025-11-13T17:49:23.064865',
    'updated': '2025-11-13T17:49:23.064888',
    'due_date': None,
    'comments': [],
    'components': []}},
  {'id': 'ISSUE-5',
   'fields': {'project': 'MOBILE',
    'summary': 'App freezes on loading user profile',
    'issuetype': 'Bug',
    'description': 'The application becomes unresponsive when a user profile is loaded.',
    'priority': 'High',
    'assignee': {'name': 'musa'},
    'status': 'In Progress',
    'created': '2025-11-13T17:49:23.065128',
    'updated': '2025-11-13T17:49:23.065135',
    'due_date': None,
    'comments': [],
    'components': []}},
  {'id': 'ISSUE-6',
   'fields': {'project': 'BACKEND',
    'summary': 'Authentication servic

**Action Agent**: I found the following unresolved issues assigned to musa:

* In Progress Issues

* **ISSUE-4**: UI glitch on login screen
* **ISSUE-5**: App freezes on loading user profile
* **ISSUE-6**: Authentication service timeout

* Open issues

* **ISSUE-7**: Implement password reset feature
* **ISSUE-8**: Add push notification support

Which issues you would like to resolve?

**Simulated User**: Only the ones with High priority.

**Action Agent**: All of your In-progress issues are of high priority. Would you like to resolve all three of them?

**Simulated User**: Yes

In [ ]:
jira.perform_bulk_issue_operations(issueUpdates=[{'status': 'Resolved', 'issueId': 'ISSUE-4'}, {'issueId': 'ISSUE-5', 'status': 'Resolved'}, {'issueId': 'ISSUE-6', 'status': 'Resolved'}])

{'bulkProcessed': True,
 'updatesCount': 3,
 'successfulUpdates': ['ISSUE-4', 'ISSUE-5', 'ISSUE-6'],
 'deletedIssues': []}

**Action Agent**: I have resolved the following issues:

* **ISSUE-4**: UI glitch on login screen
* **ISSUE-5**: App freezes on loading user profile
* **ISSUE-6**: Authentication service timeout

**Simulated User**: Now, create a summary report in Confluence.

In [ ]:
# proto_ignore
import confluence

**Action Agent**: I can do that. What would you like the title of the report to be?

**Simulated User**: Instead, notify the dev team on Slack about the resolved issues with their ids along with the total count of resolved issues

In [ ]:
# proto_ignore
import slack

In [ ]:
slack.list_channels()

{'ok': True,
 'channels': [{'messages': [{'ts': '1688682784.334459',
     'user': 'U04L7NE5Q1Y',
     'text': "Welcome everyone to the marketing brainstorming session!  Let's kick off by sharing any initial campaign ideas for Q3.",
     'reactions': [{'name': 'rocket',
       'users': ['U04L7NE5Q1Y', 'U04M2R8JCQ6', 'U04M526DV51'],
       'count': 3}],
     'is_starred': True},
    {'ts': '1688683000.456789',
     'user': 'U04M2R8JCQ6',
     'text': 'I think we should focus on a social media campaign highlighting our sustainability initiatives.',
     'reactions': [{'name': 'thumbsup',
       'users': ['U04L7NE5Q1Y', 'U04M526DV51', 'U04LMCYSD2X'],
       'count': 3}],
     'is_starred': False},
    {'ts': '1688684000.987654',
     'user': 'U04LMCYSD2X',
     'text': 'Has anyone seen those interactive ads on platform X?',
     'reactions': [],
     'is_starred': False},
    {'ts': '1688685000.123456',
     'user': 'U04M2R8JCQ6',
     'text': 'Check out this great resource: https://market

In [ ]:
slack.post_chat_message(text='Hi team, 3 high-priority issues assigned to Musa have been resolved. The resolved issues are: ISSUE-4, ISSUE-5, and ISSUE-6.', channel='C356E134F')

{'ok': True,
 'message': {'channel': 'C356E134F',
  'text': 'Hi team, 3 high-priority issues assigned to Musa have been resolved. The resolved issues are: ISSUE-4, ISSUE-5, and ISSUE-6.',
  'attachments': None,
  'blocks': None,
  'ts': '1763056163.1650248',
  'as_user': None,
  'icon_emoji': None,
  'icon_url': None,
  'link_names': None,
  'markdown_text': None,
  'metadata': None,
  'mrkdwn': None,
  'parse': None,
  'reply_broadcast': None,
  'thread_ts': None,
  'unfurl_links': None,
  'unfurl_media': None,
  'username': None}}

**Action Agent**: I've sent the message to the dev-team channel.

# Final Assertion

1. Assert that all three high priority issues assigned to "musa" have been updated to status "Resolved".
2. Assert that the Slack channel "dev-team" contains a message (or messages) with all issue ids of high priority resolved issues.

In [ ]:
import jira
import slack

# Custom assertion functions to replace assertion_utils
def compare_string(actual, expected, message):
    assert actual == expected, f"{message}. Expected: {expected}, but got: {actual}"

def compare_is_string_subset(actual, expected_set, message):
    assert actual.issubset(expected_set), f"{message}. Expected subset: {expected_set}, but got: {actual}"

# 1. Assert that all three high priority issues assigned to "musa" have been updated to status "Resolved".
jql_query = "assignee = 'musa' AND priority = 'High' AND status = 'Resolved'"
search_results = jira.search_issues_jql(jql=jql_query)

# Debug: Print the returned issues to verify the query
resolved_issues = search_results.get('issues', [])
print("Resolved Issues:", resolved_issues)  # Debugging line

# If no issues found, let's check if the query is correct by looking at the full results
if not resolved_issues:
    print(f"No issues found with query: {jql_query}")

# **Correctly extract the 'id' from resolved_issues** (the issue key here is actually the 'id')
issue_keys = [str(issue.get('id')) for issue in resolved_issues if issue.get('id') is not None]
compare_string(len(issue_keys), 3, "Expected 3 high-priority resolved issues for user 'musa'.")

# 2. Assert that the Slack channel "dev-team" contains a message (or messages) with all issue ids of high priority resolved issues.
target_channel_name = "dev-team"
dev_team_channel_id = None

# Find the channel by name instead of hardcoding channel id
all_channels = slack.list_channels(types="public_channel").get('channels', [])
for channel in all_channels:
    if channel.get('name') == target_channel_name:
        dev_team_channel_id = channel.get('id')
        break

# Assert that the channel exists
compare_string(dev_team_channel_id is not None, True, "Slack channel 'dev-team' not found.")

if dev_team_channel_id:
    # Use sorted list for deterministic message order
    issue_keys_sorted = sorted(issue_keys)  # Sort for predictable order
    msg_text = (
        f"Hi team, {len(issue_keys_sorted)} high-priority issues assigned to Musa have been resolved. "
        "The resolved issues are: " + ", ".join(issue_keys_sorted)
    )

    # Send the message to the Slack channel
    slack.post_chat_message(text=msg_text, channel=dev_team_channel_id)

    # Fetch and check the Slack messages for the issue IDs (increased limit to 50 for more reliable checking)
    history_response = slack.get_conversation_history(channel=dev_team_channel_id, limit=50)
    messages = history_response.get('messages', [])

    found_issue_ids = set()
    for msg in messages:
        message_text = msg.get('text', '') or ''  # Handle missing message text safely
        for issue_id in issue_keys_sorted:
            if issue_id in message_text:
                found_issue_ids.add(issue_id)

    # Use custom assertion to check if all expected issue IDs were found
    compare_is_string_subset(found_issue_ids, set(issue_keys_sorted), "Not all resolved issue IDs were found in the 'dev-team' channel history.")


Resolved Issues: [{'id': 'ISSUE-4', 'fields': {'project': 'WEBAPP', 'summary': 'UI glitch on login screen', 'issuetype': 'Bug', 'description': 'The login button is misaligned on Firefox.', 'priority': 'High', 'assignee': {'name': 'musa'}, 'status': 'Resolved', 'created': '2025-11-13T17:49:23.064865', 'updated': '2025-11-13T17:49:23.123995', 'due_date': None, 'comments': [], 'components': []}}, {'id': 'ISSUE-5', 'fields': {'project': 'MOBILE', 'summary': 'App freezes on loading user profile', 'issuetype': 'Bug', 'description': 'The application becomes unresponsive when a user profile is loaded.', 'priority': 'High', 'assignee': {'name': 'musa'}, 'status': 'Resolved', 'created': '2025-11-13T17:49:23.065128', 'updated': '2025-11-13T17:49:23.124009', 'due_date': None, 'comments': [], 'components': []}}, {'id': 'ISSUE-6', 'fields': {'project': 'BACKEND', 'summary': 'Authentication service timeout', 'issuetype': 'Bug', 'description': 'The authentication service times out under heavy load.', 